# 真实集成：响应级语义熵

真实 API 采样完整回答，本地 NLI 完成双向等价判定、语义聚类和响应级语义熵。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai,nli]"`。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `LOCAL_NLI_MODEL: str`：标签含 entailment/contradiction 的本地 NLI 模型。
- `equivalence: str = 'official'`：官方三态等价规则。
- `base: float = 2.0`：熵单位（bits）。
- `NUM_SAMPLES: int = 5`：熵估计样本数。

## 步骤 1：配置真实生成和 NLI

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

from hallucination.detection.backends.nli import BidirectionalNLIBackend
LOCAL_NLI_MODEL = "/absolute/path/to/your/nli-model"
NLI_DEVICE = "cpu"
if not Path(LOCAL_NLI_MODEL).is_dir():
    raise ValueError("Set LOCAL_NLI_MODEL to an existing local NLI model directory.")
nli = BidirectionalNLIBackend(LOCAL_NLI_MODEL, device=NLI_DEVICE)

## 步骤 2：运行语义熵

In [ ]:
from hallucination.detection.blackbox.response_semantic_entropy import ResponseSemanticEntropyScorer
from hallucination.detection.pipelines.short_answer import BlackBoxSamplingPipeline
NUM_SAMPLES = 5
scorer = ResponseSemanticEntropyScorer(nli, include_response=True, equivalence="official", base=2.0)
result = BlackBoxSamplingPipeline(backend, scorers=[scorer], num_samples=NUM_SAMPLES).detect_prompt("请用两句话介绍蓝鲸的食性和分类。")
score = result.method_scores[0]
print(result.metadata["response"])
print(score.raw["semantic_entropy"], score.raw["num_semantic_sets"], score.raw["clusters"])

## 结果解读与限制

低熵只说明样本集中。稳定重复的错误依然可能低风险；NLI 的语言和领域有效性必须单独验证。